# Phòng thí nghiệm TTS — benchmark & serve trên Colab

Notebook này **không chứa logic**. Toàn bộ engine, metric và server nằm trong thư mục
`colab/` của repo; ở đây chỉ là bảng điều khiển. Nhờ vậy con số bạn đo ở mục 5 đúng
bằng con số pipeline nhận ở mục 6 — cùng một file code chạy cả hai đường.

| Mục | Làm gì |
|---|---|
| 1–3 | Kiểm tra GPU, nạp code, cài dependency |
| 4 | Tải checkpoint |
| **5** | **Benchmark** — 7 engine, cùng bộ câu, sinh `report.md` |
| **6** | **Serve** — mở endpoint cho pipeline dưới máy gọi lên |

> **Runtime → Change runtime type → GPU.** T4 (free tier) đủ cho mọi engine ở đây,
> chạy lần lượt từng cái. Chỉ chọn A100 nếu muốn giữ nhiều model nóng cùng lúc.

Đọc `colab/README.md` để biết vì sao chọn đúng 7 engine này và mỗi chỉ số bắt lỗi gì.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Nạp code từ repo

Chọn một trong ba đường. `git` là đường sạch nhất — nó cũng ghi lại đúng commit nào
sinh ra bảng số, thứ mà luận văn cần khi ai đó hỏi "chạy lại có ra không".

In [ ]:
SOURCE = "upload"     # "git" | "upload" | "drive"
REPO   = ""           # SOURCE="git":    https://github.com/<ban>/<repo>.git
DRIVE  = "MyDrive/MultilingualVideoDubbingSystem"   # SOURCE="drive"

import os, sys, shutil, zipfile
from pathlib import Path

WORK = Path("/content/lab")

if SOURCE == "git":
    if not REPO:
        raise ValueError("dat REPO truoc da")
    shutil.rmtree("/content/repo", ignore_errors=True)
    !git clone -q --depth 1 {REPO} /content/repo
    src = Path("/content/repo/colab")
elif SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    src = Path("/content/drive") / DRIVE / "colab"
else:
    # Duoi may: make colab-zip  ->  dist/colab.zip, roi keo tha vao day.
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(name) as zf:
        zf.extractall("/content/unzipped")
    src = next(Path("/content/unzipped").rglob("engines/tts_base.py")).parent.parent

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(src, WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("code:", WORK)
print(sorted(p.name for p in WORK.iterdir() if not p.name.startswith(".")))

## 3. Cấu hình + cài dependency

`ENGINES` quyết định cài gì. Bỏ bớt engine nào không định chạy — mỗi engine là vài
phút cài đặt và vài GB tải về.

In [ ]:
# Bo bot dong nao khong can. Xem bang day du trong colab/README.md.
ENGINES = [
    "mms",       # san: VITS mot giong, ~1100 ngon ngu
    "vixtts",    # XTTS-v2 fine-tune tieng Viet, clone
    "f5_vi",     # F5-TTS fine-tune ~1000h ViVoice, clone
    "piper",     # tran toc do, chay CPU
    "edge",      # tran chat luong, cloud, khong clone
    # "xtts_v2", # XTTS goc - 17 ngon ngu, KHONG co tieng Viet
    # "f5_base", # F5 goc en/zh - de tach "kien truc" khoi "fine-tune"
]

LANGUAGE     = "vi"
AUTH_TOKEN   = "doi-chuoi-nay-di"   # Bearer token cho muc 6. De rong = endpoint mo
DEFAULT_ENGINE = "vixtts"           # engine dung khi client khong noi ro
SAMPLE_RATE  = 24000                # phai khop TTS_SAMPLE_RATE trong .env
PORT         = 8000
RUN_METRICS  = True                 # False = chi do toc do, nhanh hon nhieu

# Phuc vu luon ASR (whisper) va dich (NLLB) o muc 6, khong chi TTS.
# Bat cai nay thi may duoi KHONG luu trong so nao ca - do la muc dich.
RUN_STAGES   = True

In [ ]:
%%capture install_log
import subprocess

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=False)

sh("pip install -q -r requirements.txt")

# ASR + dich chay o day thay vi duoi may: whisper-medium 1.4 GB va
# NLLB-200 2.3 GB la hai thu nang nhat pipeline tung tai ve laptop.
if RUN_STAGES:
    sh("pip install -q faster-whisper transformers sentencepiece")

if {"xtts_v2", "vixtts"} & set(ENGINES):
    # coqui-tts la ban fork con duoc bao tri - dung dependency ma ai-service dung.
    sh("pip install -q coqui-tts==0.27.5")
if {"f5_vi", "f5_base"} & set(ENGINES):
    sh("pip install -q f5-tts")
if "mms" in ENGINES:
    sh("pip install -q transformers")
if "piper" in ENGINES:
    sh("pip install -q piper-tts")
if "edge" in ENGINES:
    sh("pip install -q edge-tts")

In [ ]:
# Chay cell nay neu import o muc duoi bao loi.
print(install_log.stdout[-3000:])

## 4. Checkpoint + giọng tham chiếu

Engine tự tải weight ở lần `load()` đầu tiên, trừ họ XTTS: `Xtts.load_checkpoint`
đọc thẳng từ thư mục nên phải tải trước.

viXTTS **không ship `speakers_xtts.pth`**. Đó là bảng speaker embedding dùng chung,
không phải trọng số đã fine-tune, nên lấy từ XTTS-v2 gốc là đúng chứ không phải chắp vá.

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

CKPT = Path("checkpoints"); CKPT.mkdir(exist_ok=True)

if "vixtts" in ENGINES:
    d = CKPT / "vixtts"
    snapshot_download("capleaf/viXTTS", local_dir=str(d),
                      allow_patterns=["config.json", "model.pth", "vocab.json", "samples/*"])
    if not (d / "speakers_xtts.pth").exists():
        import shutil
        shutil.copy(hf_hub_download("coqui/XTTS-v2", "speakers_xtts.pth"),
                    d / "speakers_xtts.pth")
    print("vixtts:", sorted(p.name for p in d.iterdir()))

if "xtts_v2" in ENGINES:
    d = CKPT / "xtts_v2"
    snapshot_download("coqui/XTTS-v2", local_dir=str(d),
                      allow_patterns=["config.json", "model.pth", "vocab.json",
                                      "speakers_xtts.pth"])
    print("xtts_v2:", sorted(p.name for p in d.iterdir()))

### Giọng tham chiếu

Các engine clone cần một file wav mẫu. **Đổi sang giọng thật trong clip bạn đang
dub** để SECS ở mục 5 nói lên điều gì đó — với giọng mẫu có sẵn thì cột đó chỉ đo
được "engine có clone hay không", không đo được "clone tốt tới đâu".

6–15 giây thoại sạch, một người nói, không nhạc nền.

In [ ]:
import IPython.display as ipd

REFERENCE = hf_hub_download("capleaf/viXTTS", "samples/nu-luu-loat.wav")

# Thay bang giong that:
#   from google.colab import files; REFERENCE = next(iter(files.upload()))
# hoac lay tu chinh job cua ban:
#   data/jobs/<job_id>/references/SPEAKER_00.wav

print("reference:", REFERENCE)
ipd.display(ipd.Audio(REFERENCE))

## 5. Benchmark

Chạy từng engine trên cùng bộ câu, đo RTF / WER / CER / SECS / MOS, ghi ra
`results/`. Engine được unload giữa các lần nên T4 không OOM.

Bộ câu xếp **từ 1 từ đến 28 từ** là có chủ ý: model card của viXTTS ghi rõ nó yếu ở
câu dưới 10 từ, mà phụ đề phim thì đa số dưới 10 từ. Bảng "WER theo độ dài câu" trong
báo cáo chính là bảng quyết định engine nào dùng được.

In [ ]:
import benchmark, importlib, sentences
importlib.reload(sentences); importlib.reload(benchmark)

argv = ["--language", LANGUAGE, "--reference", REFERENCE,
        "--engines", ",".join(ENGINES), "--out", "results"]
if not RUN_METRICS:
    argv.append("--no-metrics")

benchmark.main(argv)

### Nghe lại

Số liệu không đo được ngữ điệu. Lấy hai engine đứng đầu bảng và nghe — nhất là bốn
câu ngắn nhất, chỗ mà lỗi "đuôi câu lạ" của viXTTS lộ ra.

In [ ]:
import IPython.display as ipd
from pathlib import Path
import sentences as S

LISTEN = ENGINES[:3]          # doi thanh hai engine dung dau bang
LINES  = [0, 1, 2, 3, 7]      # 4 cau ngan nhat + 1 cau dai de doi chung

for idx in LINES:
    line = S.get(LANGUAGE)[idx]
    print(f"\n[{line.words} tu] {line.text}\n   -> {line.probe}")
    for name in LISTEN:
        wav = Path("results/audio") / name / f"line_{idx:02d}.wav"
        if wav.exists():
            print("  ", name)
            ipd.display(ipd.Audio(str(wav)))

In [ ]:
# Tai ket qua ve may de dua vao luan van.
import shutil
from google.colab import files
shutil.make_archive("/content/tts_results", "zip", "results")
files.download("/content/tts_results.zip")

## 6. Serve — mở endpoint cho pipeline dưới máy

Từ đây trở xuống độc lập với mục 5. Server phục vụ **mọi engine** trong registry;
client chọn engine theo từng request, nên A/B trên một job thật chỉ là đổi một biến
môi trường.

Mỗi lúc chỉ một model nằm trong VRAM — đổi engine thì model cũ được unload trước.

In [ ]:
import os, threading, uvicorn, importlib

os.environ.update(
    AUTH_TOKEN=AUTH_TOKEN,
    DEFAULT_ENGINE=DEFAULT_ENGINE,
    SAMPLE_RATE=str(SAMPLE_RATE),
    REF_DIR="/content/refs",
    OUT_DIR="/content/out",
    # ALLOW_MULTI_RESIDENT="1",   # chi bat tren A100
)

import server; importlib.reload(server)

threading.Thread(
    target=lambda: uvicorn.run(server.app, host="0.0.0.0", port=PORT, log_level="warning"),
    daemon=True,
).start()

import time; time.sleep(3)
!curl -s localhost:{PORT}/health -H "Authorization: Bearer {AUTH_TOKEN}" | head -c 600

### Tunnel ra ngoài

`cloudflared` — không cần tài khoản, không cần token. URL ngẫu nhiên **đổi mỗi lần
chạy lại cell**, nên đây là dòng phải cập nhật lại trong `.env` sau mỗi lần restart.

In [ ]:
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

import re, subprocess, threading, time

PUBLIC_URL = None
proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

def _watch():
    global PUBLIC_URL
    for line in proc.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m and PUBLIC_URL is None:
            PUBLIC_URL = m.group(0)

threading.Thread(target=_watch, daemon=True).start()
for _ in range(40):
    if PUBLIC_URL:
        break
    time.sleep(1)

print("public URL:", PUBLIC_URL)

In [ ]:
# Dan khoi nay vao ai-service/.env roi khoi dong lai AI service.
print(f"""
REMOTE_URL={PUBLIC_URL}
REMOTE_TOKEN={AUTH_TOKEN}
REMOTE_TTS_ENGINES={','.join(ENGINES)}
REMOTE_TTS_LANGUAGES={LANGUAGE}
REMOTE_ASR_ENABLED={'true' if RUN_STAGES else 'false'}
REMOTE_TRANSLATION_ENABLED={'true' if RUN_STAGES else 'false'}
""".strip())

if RUN_STAGES:
    print("\n-> ASR + dich chay tren Colab. Thu muc models/ duoi may se rong.")
    print("   Luu y: bat REMOTE_ASR_ENABLED nghia la AUDIO (da nen Opus) duoc")
    print("   upload len day, khong con chi text nua.")

In [ ]:
# Kiem tra qua tunnel truoc khi dan.
!curl -s {PUBLIC_URL}/engines -H "Authorization: Bearer {AUTH_TOKEN}" | head -c 800; echo

## 7. Nối vào pipeline

1. Dán khối ở trên vào `ai-service/.env`.
2. Khởi động lại AI service (`./scripts/run-native.sh`) — settings đọc lúc khởi động.
3. Kiểm tra:
   ```bash
   curl -s localhost:47800/health/models | python3 -m json.tool
   ```

**Một endpoint, ba stage.** Notebook này phục vụ cả `/transcribe`, `/translate` và
`/synthesize`. Đó là lý do máy dưới không cần lưu trọng số nào:

| Model | Dung lượng nếu chạy local | Ở đâu bây giờ |
|---|---|---|
| whisper-medium | 1.4 GB | Colab |
| nllb-200-distilled-600M | 2.3 GB | Colab |
| viXTTS / F5-TTS | 2.5–3 GB | Colab |

**Đánh đổi phải nói rõ.** Bật `REMOTE_ASR_ENABLED` nghĩa là **dải tiếng nói của video
được upload** (đã nén Opus ~24 kbps — phim 10 phút ≈ 1.7 MB thay vì 19 MB wav). Video
thì vẫn không bao giờ rời máy, nhưng câu "chỉ có text rời khỏi máy" không còn đúng.
Dịch thì chỉ gửi text nên không có đánh đổi nào.

**So sánh engine trên job thật:** chạy cùng một job hai lần với `force_model` khác nhau
rồi so `synthesis_manifest`. Cột `duration_ratio` cho biết engine nào khớp slot thời gian
tốt hơn — thứ benchmark câu rời không đo được vì nó không có slot nào để khớp.

**Khi tunnel chết** — Colab timeout, mất mạng — client thử lại 3 lần. TTS rơi xuống
MMS-TTS; ASR và dịch thì rơi về model local, tức là **sẽ tải model về máy** đúng lần đó.
Muốn máy tuyệt đối không có model thì tắt AI service khi Colab chết thay vì để job chạy.